# 🌐 Red Bayesiana: Diagnóstico Médico
Este notebook construye y analiza una red bayesiana sencilla para modelar un problema de diagnóstico médico.

## 📥 Instalación y librerías necesarias

In [1]:
!pip install pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 12.5 MB/s eta 0:00:00


## ⚙️ Definición de la red bayesiana
Modelaremos un escenario médico:
- **Fiebre** depende de si hay `Infección`.
- **Tos** depende de `Infección` y `Alergia`.
- **Dolor de cabeza** depende de `Fiebre`.


In [2]:

from pgmpy.models import BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Definimos la estructura de la red
modelo = BayesianNetwork([
    ('Infeccion', 'Fiebre'),
    ('Infeccion', 'Tos'),
    ('Alergia', 'Tos'),
    ('Fiebre', 'DolorCabeza')
])
modelo


INFO:numexpr.utils:Note: NumExpr detected 12 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.


ImportError: BayesianNetwork has been deprecated. Please use DiscreteBayesianNetwork instead.

## 📊 Definición de las probabilidades condicionales

In [ ]:

# CPD: Infección (probabilidad base)
cpd_infeccion = TabularCPD(variable='Infeccion', variable_card=2,
                           values=[[0.9], [0.1]],  # 90% no, 10% sí
                           state_names={'Infeccion': ['No', 'Si']})

# CPD: Alergia
cpd_alergia = TabularCPD(variable='Alergia', variable_card=2,
                         values=[[0.7], [0.3]],  # 70% no, 30% sí
                         state_names={'Alergia': ['No', 'Si']})

# CPD: Fiebre depende de Infección
cpd_fiebre = TabularCPD(variable='Fiebre', variable_card=2,
                        values=[[0.95, 0.2],   # No fiebre
                                [0.05, 0.8]],  # Sí fiebre
                        evidence=['Infeccion'],
                        evidence_card=[2],
                        state_names={'Fiebre': ['No', 'Si'],
                                     'Infeccion': ['No', 'Si']})

# CPD: Tos depende de Infección y Alergia
cpd_tos = TabularCPD(variable='Tos', variable_card=2,
                     values=[[0.9, 0.6, 0.7, 0.1],  # No tos
                             [0.1, 0.4, 0.3, 0.9]], # Sí tos
                     evidence=['Infeccion', 'Alergia'],
                     evidence_card=[2, 2],
                     state_names={'Tos': ['No', 'Si'],
                                  'Infeccion': ['No', 'Si'],
                                  'Alergia': ['No', 'Si']})

# CPD: Dolor de cabeza depende de Fiebre
cpd_dolor = TabularCPD(variable='DolorCabeza', variable_card=2,
                       values=[[0.8, 0.3],  # No dolor
                               [0.2, 0.7]], # Sí dolor
                       evidence=['Fiebre'],
                       evidence_card=[2],
                       state_names={'DolorCabeza': ['No', 'Si'],
                                    'Fiebre': ['No', 'Si']})

# Agregar CPDs al modelo
modelo.add_cpds(cpd_infeccion, cpd_alergia, cpd_fiebre, cpd_tos, cpd_dolor)

# Validar modelo
modelo.check_model()


## 🔍 Inferencia en la red

In [ ]:

inferencia = VariableElimination(modelo)

# Ejemplo: probabilidad de tener infección si hay fiebre y tos
resultado = inferencia.query(variables=['Infeccion'], evidence={'Fiebre': 'Si', 'Tos': 'Si'})
print(resultado)
